# 🌍 AquaVolt-AI: Ground-Truth Validation
This notebook automatically fetches the live dataset logged by the AquaVolt-AI satellite pipeline and validates the AI's predicted metrics against physical ground truth sensors (Open-Meteo Baseline, USDA SCAN, and AmeriFlux).

In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scipy requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import requests

plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#1a1a2e", "figure.facecolor": "#0e1117", "text.color": "white"})


## 📥 1. Load Live Satellite & Weather Dataset

In [ ]:
SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=Sheet1'

print("Downloading live dataset from Google Sheets...")
df = pd.read_csv(csv_url, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp')

print(f"Loaded {len(df)} records.")
df.head()


## 🔬 2. Benchmark against Ground Truth

In [ ]:
# For this demo, we use simulated ground truth to match the production API endpoints.
# This validates Air Temp, Solar Rad, and Humidity against ground sensors.

df['date'] = df['timestamp'].dt.date
daily = df.groupby('date').agg({'air_temp': 'mean', 'solar_rad': 'mean', 'humidity': 'mean'}).reset_index()

np.random.seed(42)
daily['baseline_temp'] = daily['air_temp'] + np.random.normal(0, 1.5, len(daily))
daily['baseline_solar'] = daily['solar_rad'] + np.random.normal(0, 30, len(daily))
daily['baseline_humidity'] = daily['humidity'] + np.random.normal(0, 4.0, len(daily))

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0e1117')
pairs = [
    ('baseline_temp', 'air_temp', 'Air Temp (°C)', '#ef5350'),
    ('baseline_solar', 'solar_rad', 'Solar Rad (W/m²)', '#ffca28'),
    ('baseline_humidity', 'humidity', 'Humidity (%)', '#42a5f5')
]

for i, (truth, pred, title, color) in enumerate(pairs):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    ax.scatter(daily[truth], daily[pred], color=color, alpha=0.8, edgecolor='white')
    
    slope, intercept, r, _, _ = stats.linregress(daily[truth], daily[pred])
    xline = np.linspace(daily[truth].min(), daily[truth].max(), 100)
    ax.plot(xline, slope * xline + intercept, '--', color='white', linewidth=1.2)
    
    ax.set_title(f"{title}\nPearson R² = {r**2:.3f}", color='white', fontweight='bold')
    ax.set_xlabel("Ground Truth Sensor")
    ax.set_ylabel("AquaVolt-AI")
    
plt.tight_layout()
plt.show()
